<div align="right"><sub>Notebook 最終更新: 2026-03-24 21:37</sub></div>
<h1><strong>05. AIエージェントの基礎</strong></h1>

今回からは，LLMを単体で使うのではなく，役割を持った「エージェント」として組み合わせて，複雑なタスクをこなす方法を学びます．
まずは，回答を行う **Executor** と，その回答をチェックする **Critic** の2役を組み合わせた「自己修正ループ」を体験しましょう．

### この Notebook の構成
| セクション | 内容 | ねらい |
|:--|:--|:--|
| 1 | チャット関数の準備 | 共通部品 |
| 2 | 簡単なタスクでエージェントを試す | **成功例** — Executor-Critic が正しく動作する様子 |
| 3 | 難しいタスクでエージェントを試す | **限界の可視化** — 3Bモデルで破綻する様子 |
| 4 | プロンプト改善の効果 | プロンプトの工夫でどこまで改善できるか |
| 5 | 考察・まとめ | モデル能力とエージェントパターンの関係 |

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
print('永続ディレクトリ:', PERSIST_ROOT)
from src.common import load_llm, generate_text
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

model, tokenizer = load_llm()
print('準備完了')


## **1. チャット関数の準備**
日本語LLMのチャット機能を使って，システムプロンプトを受け取れるラッパー関数を作成します．

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

## **2. 簡単なタスクでエージェントを試す（成功例）**

まずは3Bモデルでも確実にこなせる **制約の少ないタスク** で，Executor-Critic パターンが正しく動作する様子を観察します．

ここでは「カジュアルな文章を丁寧語（です・ます調）に変換する」というシンプルなタスクを使います．

In [ ]:
# --- 簡単なタスク用のプロンプト設定 ---
easy_executor = RoleConfig(
    name="Executor",
    system_prompt="あなたは丁寧な文章作成者です。必ず日本語で回答してください。指示に従い、変換後の文章のみを出力してください。余計な説明は不要です。"
)
easy_critic = RoleConfig(
    name="Critic",
    system_prompt="あなたは日本語文章の校閲者です。必ず日本語で回答してください。回答が以下の条件を満たしているか確認し、結果を報告してください。\n(1) です・ます調になっているか\n(2) 元の情報が欠落していないか\n(3) 不要な情報が追加されていないか\n問題がなければ「誤りなし」とだけ答えてください。問題がある場合は、どの条件を満たしていないかを簡潔に指摘してください。"
)

agent_easy = LLMExecutorCriticAgent(llm_chat, role_configs=[easy_executor, easy_critic])

easy_query = "次の文章をです・ます調に書き直してください。元の情報はすべて残すこと。\n\n文章: 来週の月曜、朝10時に会議室Aに集合。資料は各自で印刷して持ってきて。遅刻厳禁。" #@param{type:'string'}

answer, log, steps = agent_easy.run_pipeline(easy_query)

print("=== エージェントの処理過程 ===")
print(log)
print("\n=== 最終回答 ===")
print(answer)

### ✅ 観察ポイント
- Executor が「です・ます調」に変換した回答を出力できていますか？
- Critic は正しく評価し，問題なければ「誤りなし」と回答していますか？
- もし Critic が改善点を指摘した場合，修正後の回答は改善されていますか？

このように，**制約が少なく明確なタスク** では，3Bモデルでも Executor-Critic パターンが機能します．

## **3. 難しいタスクでエージェントを試す（限界の可視化）**

次に，**複数の制約を同時に満たす必要がある難しいタスク** で同じパターンを試します．

5つの条件（日時・場所・参加条件・文体・文字数制限）を同時に満たす案内文の作成です．

In [ ]:
agent_default = LLMExecutorCriticAgent(llm_chat)

hard_query = "次の案内文を、(1) 日時、(2) 場所、(3) 参加条件、(4) です・ます調、(5) 80字以内、の5条件を満たすように書き直してください。案内文: 研究室見学をする予定です。来たい人は参加できますが、申込みした人を優先します。今週土曜日の午後2時からで、場所は情報学部1号館3階の301室です。" #@param{type:'string'}

answer_hard, log_hard, _ = agent_default.run_pipeline(hard_query)

print("=== エージェントの処理過程 ===")
print(log_hard)
print("\n=== 最終回答 ===")
print(answer_hard)
print(f"\n文字数: {len(answer_hard)}字")

### ⚠️ 問題点を確認しましょう
以下の点をチェックしてください：

| チェック項目 | 結果 |
|:--|:--|
| Executor は5条件をすべて満たしているか？ | |
| Critic は的確な指摘をしているか？ | |
| Critic がハルシネーション（実際にはある情報を「ない」と言う）をしていないか？ | |
| 80字以内という制約は守られているか？ | |
| 修正後に実際に改善されているか？ | |

おそらく，いくつかの問題が見つかるはずです．次に，反復回数を増やしても改善するかを確認します．

In [ ]:
# --- 反復回数を3回に増やして試す ---
answer_iter, log_iter, _ = agent_default.run_pipeline(hard_query, max_iterations=3)

print("=== 3回反復の処理過程 ===")
print(log_iter)
print("\n=== 最終回答 ===")
print(answer_iter)
print(f"\n文字数: {len(answer_iter)}字")

### 🔍 反復の効果
- 反復回数を増やすことで回答は改善されましたか？
- それとも，同じ問題が繰り返されたり，新たな問題が生じたりしていませんか？

3Bモデルでは，**Critic 自身のメタ認知能力（自分の回答を客観的に評価する能力）が弱い** ため，反復しても改善に限界があります．

## **4. プロンプト改善の効果**

プロンプトを工夫すれば，同じ3Bモデルでも結果はどこまで改善できるでしょうか？

ここでは，**Executor にテンプレート形式の出力指示**，**Critic にチェックリスト形式の評価指示** を与えてみます．

In [ ]:
improved_executor = RoleConfig(
    name="Executor",
    system_prompt="あなたは丁寧な文章作成者です。必ず日本語で回答してください。元の情報を保ちながら、読みやすく簡潔に書き直してください。回答は書き直した文章のみを出力し、複数のバージョンや解説は不要です。" #@param{type:'string'}
)
improved_critic = RoleConfig(
    name="Critic",
    system_prompt="あなたは文章の校閲者です。必ず日本語で回答してください。以下の5項目を1つずつチェックし、各項目に ✓（満たしている）または ✗（満たしていない）を付けて報告してください。\n(1) 日時が含まれているか\n(2) 場所が含まれているか\n(3) 参加条件が含まれているか\n(4) です・ます調になっているか\n(5) 80字以内か（文字数を数えて報告すること）\nすべて ✓ なら「誤りなし」と最後に追記してください。✗ がある場合は最小限の修正案を提示してください。元の文に書かれていない情報を「不足」と指摘しないでください。" #@param{type:'string'}
)

agent_improved = LLMExecutorCriticAgent(llm_chat, role_configs=[improved_executor, improved_critic])

answer_improved, log_improved, _ = agent_improved.run_pipeline(hard_query, max_iterations=2)

print("=== 改善プロンプトでの処理過程 ===")
print(log_improved)
print("\n=== 最終回答 ===")
print(answer_improved)
print(f"\n文字数: {len(answer_improved)}字")

### 📊 比較
セクション3（デフォルトプロンプト）とセクション4（改善プロンプト）の結果を比較してみましょう：

| 観点 | デフォルト | 改善版 |
|:--|:--|:--|
| Executor の出力形式 | | |
| Critic の指摘の正確さ | | |
| 文字数制限の遵守 | | |
| 最終回答の5条件充足 | | |

プロンプトの工夫で改善される部分はありますが，タスクの複雑さがモデルの能力を超えている場合は，根本的な解決にはなりません．

## **5. 考察・まとめ**

### エージェントパターンの有効性とモデル能力の関係

今回の実験を通じて，以下の知見が得られます：

1. **Executor-Critic パターンは有効** — 制約が少ないタスク（セクション2）では，3Bモデルでも自己修正ループが正しく動作しました．

2. **モデル能力がボトルネック** — Critic の役割は「他の回答を評価する」というメタ認知的なタスクであり，Executor より高い能力を要求します．3Bモデルでは：
   - 複数制約を同時に検証することが困難
   - ハルシネーション（存在する情報を「ない」と言う）が発生する
   - 文字数のカウントが不正確

3. **プロンプトエンジニアリングの効果と限界** — チェックリスト形式のプロンプトなど工夫の余地はありますが，モデルの根本的な能力上限を超えることはできません．

4. **実運用での示唆** — 大規模モデル（GPT-4, Claude 3.5 Sonnet 等）では Critic の精度が飛躍的に向上し，このパターンが真価を発揮します．モデルサイズとタスク難易度のバランスが重要です．

### まとめ
- 1つのプロンプトで完璧な回答を求める（Zero-shot）よりも，役割を分けて「自分で自分のミスを直す」プロセスを入れることで，より信頼性の高い回答が得られるようになります．
- ただし，この効果は **モデルの能力に依存** します．特に Critic には高い推論力が必要であり，小規模モデルではタスクの複雑さに応じて限界があります．
- **タスクの難易度をモデルの能力に合わせて調整する**，あるいは **より大規模なモデルを使う** ことが実践上の重要なポイントです．